Слова про антиферромагнетизм и как меняется при этом гамильтониан, его физика. Ожидаемый переход при увеличении взаимодействия

In [2]:
import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
from ed_solver import EDSolverCore

/home/quantum_wow/Документы/Obsidian Vault/Physical edu/Collective behavior/DMFT/triqs_ED-DMFT/ed_solver/ed_solver/triqs_wrapper.py:4: FutureWarning: The triqs.gf module has been renamed to triqs.gfs. Please update your imports. triqs.gf will be removed in a future version.
  from triqs.gf import *


Starting serial run at: 2026-08-27 20:37:18.393563


In [3]:
def make_AF_flat_dispersion(size_x, size_y, hopping):
    """
    This function make energy dispersion for reduced Briollien zone because of antiferromagnetism.
    """
    kx, ky = (
        np.linspace(-np.pi, np.pi, size_x, endpoint=False),
        np.linspace(-np.pi / 2, np.pi / 2, size_y // 2, endpoint=False),
    )

    kx_mesh, ky_mesh = np.meshgrid(
        kx, ky, indexing='ij'
    )

    xi_k = -2 * hopping * (
        np.cos(kx_mesh) + np.cos(ky_mesh)
    )

    return xi_k.flatten()

In [4]:
def make_init_hybridization(beta, freq_number):
    """
    This function make initial guess of hybridization function from free case.
    :param beta: float inverse temperature;
    :param freq_number: number of Matsubara frequencies;

    Return - 1d-array with shape (freq_number, );
    """
    Matsubara_freq = 1j * np.pi * (2 * np.arange(0, freq_number) + 1) / beta

    Delta = -4j * np.sqrt(np.abs(Matsubara_freq) ** 2 + 1) ** (-1)

    return Delta

Текст про то как теперь считается локальная функция Грина при антиферромагнетизме.

**Задача**: напишите функцию, которая расчитает локальную функцию Грина с случае антиферромагнетизма.

In [5]:
def compute_local_AF_GF(flat_dispersion, G_imp_up, G_imp_down, hybr_up, hybr_down):
    """
    This function calculate local antiferromagnetic Greens function.
    """
    lattice_size = len(flat_dispersion)
    norm = 1 / lattice_size

    freq_number = len(hybr_up)

    GF_loc = np.zeros(
        (2, freq_number), dtype=np.complex128
    )

    zeta_up = G_imp_up ** (-1) + hybr_up
    zeta_down = G_imp_down ** (-1) + hybr_down

    denominator_up = (
        zeta_up[:, None] - (
            (flat_dispersion ** 2)[None, :] / zeta_down[:, None]
        )
    ) ** (-1)

    denominator_down = (
        zeta_down[:, None] - (
            (flat_dispersion ** 2)[None, :] / zeta_up[:, None]
        )
    ) ** (-1)

    GF_loc_up = norm * np.sum(denominator_up, axis=-1)
    GF_loc_down = norm * np.sum(denominator_down, axis=-1)

    GF_loc[0, :], GF_loc[1, :] = GF_loc_up, GF_loc_down

    return GF_loc

В нотбуке по функциям Грина и $\text{DMFT}$, мы написали функцию, реализующую $\text{DMFT}$-петлю. 

**Вопрос**: что нам нужно поменять в этой функции, чтобы делать $\text{DMFT}$ самосогласование для антиферромагнитного случая?

In [6]:
def dmft_loop_af(size_x:int, size_y:int, hopping:float, Delta_init_up, Delta_init_down, chemical, beta, bath_number, freq_number, 
                   interaction, h_mag, max_iter, atol, rtol, mix):
    """
    This function make self-consistent DMFT-loop for antiferromagnetic case on base of solver from EdSolverCore;

    :param size_x, size_y: integer sizes of lattice;
    :param hopping: float parameter of hopping;
    :param Delta_up (Delta_down): 1d-array of initial guess of hybridization with shape (freq_number, );
    :param chemical: THE VALUE IS MEASURED FROM HALF-FILLING, I.E. MU=0 IS MU=U/2 IN USUAL UNITS;
    :param beta: float inverse temperature;
    :param bath_number: maximal number of bath levels;
    :param freq_number: number of Matsubara frequencies;
    :param interaction: value of interaction U;
    :param h_mag: value of external Zeeman magnetic field;
    :param max_iter: number of possible iterations;
    :param atol: value of absolute tolerance;
    :param rtol: value of relative tolerance;
    :param mix: value of iteration mixing;
    """

    # 1. Initialize solver class
    solver = EDSolverCore(
        Beta=beta,
        Nbath_max=bath_number,
        Nw=freq_number
    )

    # 2. Initialize Matsubara and energies
    Matsubara_freq = 1j * np.pi * (2 * np.arange(0, freq_number) + 1) / beta

    flat_energies = make_AF_flat_dispersion(
        size_x, size_y, hopping
    )
    
    converged = False
    bar = tqdm(range(max_iter), 
        desc=f"Performing DMFT for T = {beta ** (-1):.4f}, U = {interaction:.4f} ...",
        leave=False)

    G_up_error, G_down_error = (
        np.zeros_like(Matsubara_freq, dtype=float),
        np.zeros_like(Matsubara_freq, dtype=float)
    )

    Delta_up, Delta_down = (
        Delta_init_up, Delta_init_down
    )

    plt.ion()

    fig, (ax_error, ax_bath_en, ax_bath_t) = plt.subplots(1, 3)

    for idx in bar:

        # 3.A. Solve impurity problem
        solver.solve(
            U=interaction,
            Delta_up=Delta_up,
            Delta_down=Delta_down,
            N_bath=bath_number,
            h_loc=h_mag,
            mu_loc=chemical
        )

        # 3.B. Extract impurity GF and calculate local GF
        G_imp_up, G_imp_down = solver.G_up, solver.G_down

        G_loc_up, G_loc_down = compute_local_AF_GF(
            flat_energies, G_imp_up, G_imp_down, 
            Delta_up, Delta_down
        )
                
        G_up_error = np.abs(G_imp_up - G_loc_up)
        G_down_error = np.abs(G_imp_down - G_loc_down)

        ax_error.clear()
        ax_error.plot(Matsubara_freq.imag, G_up_error, marker="o", linestyle='--', label=r'$G_{\uparrow}$', markersize=5, linewidth=2.0, color='#0501DA')
        ax_error.plot(Matsubara_freq.imag, G_down_error, marker="s", linestyle='--', label=r'$G_{\downarrow}$', markersize=5, linewidth=2.0, color="#1AAD26")
        ax_error.set_xlabel(r"$\omega_n$", fontsize=20)
        ax_error.set_ylabel(r"$|G_{\text{imp}} - G_{\text{loc}}|$", fontsize=20)
        ax_error.grid(True)

        ax_error.relim()
        ax_error.autoscale_view()
        ax_error.legend(loc='upper right', fontsize=20)

        ax_error.set_title(
            f"Iteration {idx + 1}, "
            f"max error = {max(max(G_up_error), max(G_down_error)):.3e}"
        )

        ax_bath_en.clear()
        ax_bath_en.plot(range(bath_number), np.sort(solver.eu), marker="o", linestyle='--', label=r"$\epsilon_\uparrow$", markersize=7, linewidth=2.0, color='#0501DA')
        ax_bath_en.plot(range(bath_number), np.sort(solver.ed), marker="s", linestyle='--', label=r"$\epsilon_\downarrow$", markersize=7, linewidth=2.0, color="#1AAD26")
        ax_bath_en.legend(loc='upper right', fontsize=20)
        ax_bath_en.set_xlabel(r"Bath index, $N$", fontsize=20)
        ax_bath_en.set_ylabel(r"Bath parameters, $\epsilon_l$", fontsize=20)
        ax_bath_en.grid(True)

        ax_bath_t.clear()
        ax_bath_t.plot(range(bath_number), np.sort(solver.t2u), marker="o", linestyle='--', label=r"$t_{2\uparrow}$", markersize=7, linewidth=2.0, color='#0501DA')
        ax_bath_t.plot(range(bath_number), np.sort(solver.t2d), marker="s", linestyle='--', label=r"$t_{2\downarrow}$", markersize=7, linewidth=2.0, color="#1AAD26")
        ax_bath_t.legend(loc='upper right', fontsize=20)
        ax_bath_t.set_xlabel(r"Bath index, $N$", fontsize=20)
        ax_bath_t.set_ylabel(r"Bath parameters, $t_l$", fontsize=20)
        ax_bath_t.grid(True)


        fig.tight_layout()
        fig.canvas.draw()
        fig.canvas.flush_events()

        # 3.C. Check self-consistensy
        if np.allclose(G_imp_up, G_loc_up, rtol, atol) and np.allclose(G_imp_down, G_loc_down, rtol, atol):

            converged = True
            print('\n DMFT loop is converged!')
            break

        else:
        # 3.C. Update hybridization
            
            Delta_up += mix * (G_imp_up ** (-1) - G_loc_up ** (-1))
            Delta_down += mix * (G_imp_down ** (-1) - G_loc_down ** (-1))

    plt.ioff()
    plt.show(block=False)
    plt.pause(0.5)
    plt.close('all')

    if not converged:

        msg = (
            f"\nWarning: DMFT cycle did not converge after {max_iter} iterations.\n" +
            f"  T = {beta ** (-1):.4f}, U = {interaction:.4f}\n" +
            f"  Last difference spin up   (max |G_loc - G_imp|): {G_up_error:.2e}\n" +
            f"  Last difference spin down (max |G_loc - G_imp|): {G_down_error:.2e}\n"
        )

        warnings.warn(msg, RuntimeWarning, stacklevel=2)

    return solver 

Текст про константу Кюри и возможность увидеть переход парамагнетик - антиферромагнетик, которого не должно быть в 2D из-за теоремы Мермина-Вагнера.

**Задача**: напишите функцию, которая рассчитывает константу Кюри.

In [7]:
def compute_Curie_constant(size_x, size_y, hopping, chemical, beta, interaction, dh, atol, rtol, mix):
    """
    This function calculate Curie constant from DMFT calculations.

    :param physics: para, ferro, antiferro;
    :param size_x, size_y: integer sizes of lattice;
    :param hopping: float parameter of hopping;
    """

    # In this function you can change number of frequencies for better convergence conditions
    # but physically it doesn't influence on the magnetization computation

    freq_number = 200
    hybr_init_up = make_init_hybridization(
        beta, freq_number
    )

    hybr_init_down = hybr_init_up.copy()

    result_hdh = dmft_loop_af(
        size_x, size_y, hopping, hybr_init_up, hybr_init_down, 
        chemical, beta, 4, freq_number, interaction, dh, 100, atol, rtol, mix
    )

    magnetiz_hdh = result_hdh.n_up() - result_hdh.n_down()

    chi = magnetiz_hdh / dh

    Curie_constant = chi / beta

    return Curie_constant

Построим теперь график для того, чтобы пронаблюдать фазовый переход в $\text{DMFT}$ в диапазоне $\beta \in [2.0, 9.0]$.

In [8]:
def plot_Curie_constant(size_x, size_y, hopping, chemical, interaction, dh, beta_min, beta_max, beta_step, atol, rtol, mix):
    """
    This function plot Curie constant for different betas.
    """
    betas = np.arange(beta_min, beta_max, beta_step)

    Curie_arr = np.zeros_like(betas)

    for idx, beta in enumerate(betas):

        Curie_arr[idx] = compute_Curie_constant(
            size_x, size_y, hopping, chemical, 
            beta, interaction, dh, atol, rtol, mix
        )

    plt.plot(betas, Curie_arr, label=r'$C$',marker="o", linestyle='--', markersize=5, linewidth=2.0, color='#0501DA')
    plt.xlabel(r"$\beta$", fontsize=20)
    plt.ylabel(r"$C= \chi / \beta$", fontsize=20)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.grid()
    plt.legend(fontsize=20)
    plt.show()

In [9]:
size_x, size_y = 8, 8
hopping, chemical, interaction = 1.0, 0.0, 2.0
dh = 5e-3

# here we plot Curie constant